# <span style="color:#F57C00">Exploratory Data Analysis: What the District Tables Tell Us Before We Build the Model</span>

We now have six tables, all joined through the **district code** (`district_code`; district names repeat, e.g. there are 100 districts called like another one, so we never join on names):

| Table | One row per | Years |
| :--- | :--- | :--- |
| greenness (Landsat) | district and year | 2013-2025, July + August |
| climate (ERA5) | district and month | 2000-2025 |
| population (TÜİK) | district and year | 2013-2025 |
| building permits (TÜİK) | district and month | 2013-2025 |
| static table (GHSL, SRTM) | district | - |
| SEGE-2022 | district | 2022 only |

The model will predict the **summer surface temperature** of the built-up area of a district (Landsat). This notebook answers four questions that decide how the model must be trained and tested. Every part can be run on its own after the setup, and ends with a short **What we learned**.

| Part | Question |
| :--- | :--- |
| **A2** | Which districts are left out because their satellite value is not reliable, and does this bias the picture? |
| **B2** | How do green share and surface temperature behave over the years? Is the 2025 drop real? Is there a break when Landsat 9 starts in 2022? |
| **C1** | Which variables go together with the surface temperature, in raw numbers and *inside a city*? |
| **C2** | Where does the variation live: between cities, between the districts of a city, or over the years? |

**Two words used a lot.** *Reliable* row = a district-year with a built-up area of at least 0.5 km² and with a clear view in at least half of its built-up pixels. *Within city* = a value minus the mean of the same city in the same year: what is left is the difference between the districts of one city, so weather and regional climate drop out. This is the part that parks and buildings can change.

# <span style="color:#F57C00">Setup: Load the Tables and Build One Analysis Table</span>

We read the processed tables and build **one table with one row per district and summer** (973 districts x 13 years). The rule: each source table is aggregated to this level first and then joined on `district_code` (and `year`). `validate=` makes pandas stop with an error if a join would repeat rows.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, LeaveOneGroupOut, cross_val_predict

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

PROCESSED = Path("../data/processed")

green = pd.read_csv(PROCESSED / "greenness_districts_summer_2013_2025.csv")
climate = pd.read_csv(PROCESSED / "climate_districts_monthly_2000_2025.csv")
population = pd.read_csv(PROCESSED / "tuik_districts_population_clean.csv")
permits = pd.read_csv(PROCESSED / "building_permits_districts_monthly_2013_2025.csv")
static = pd.read_csv(PROCESSED / "districts_urban_static.csv")
sege = pd.read_csv(PROCESSED / "sege_districts_2022.csv")
districts = gpd.read_file(PROCESSED / "turkey_districts_master.geojson")
cities = gpd.read_file(PROCESSED / "turkey_cities_master.geojson")

for name, table in [("greenness", green), ("climate", climate), ("population", population), ("permits", permits), ("static", static), ("sege", sege)]:
    print(f"{name:11s} {table.shape[0]:>7} rows x {table.shape[1]:>2} columns")


## Step 1: Two groupings: metropolitan cities and regions
Two simple groupings help us to read the results: the **30 metropolitan municipalities** (`city_code` = the plate number of the province; in them the TÜİK urban population is the whole population, outside them it is only the district center) and the **7 geographic regions**.

In [ ]:
METRO = {1, 6, 7, 9, 10, 16, 20, 21, 25, 26, 27, 31, 33, 34, 35, 38, 41, 42, 44, 45, 46, 47, 48, 52, 54, 55, 59, 61, 63, 65}

REGIONS = {
    "Marmara": [10, 11, 16, 17, 22, 34, 39, 41, 54, 59, 77],
    "Aegean": [3, 9, 20, 35, 43, 45, 48, 64],
    "Mediterranean": [1, 7, 15, 31, 32, 33, 46, 80],
    "Central Anatolia": [6, 18, 26, 38, 40, 42, 50, 51, 58, 66, 68, 70, 71],
    "Black Sea": [5, 8, 14, 19, 28, 29, 37, 52, 53, 55, 57, 60, 61, 67, 69, 74, 78, 81],
    "Eastern Anatolia": [4, 12, 13, 23, 24, 25, 30, 36, 44, 49, 62, 65, 75, 76],
    "Southeastern Anatolia": [2, 21, 27, 47, 56, 63, 72, 73, 79],
}
REGION_OF = {city: region for region, city_list in REGIONS.items() for city in city_list}

assert len(METRO) == 30 and len(REGION_OF) == 81, "30 metropolitan cities and 81 provinces expected"


## Step 2: Summer weather, spring rain, permits and population per district and year
- **Summer weather:** mean temperature of July + August (`temp_mean_ja`) and the number of hot days (≥ 30 °C) in July + August (`hot_days_ja`). This is the *weather of the same summer* as the satellite picture.
- **Spring rain** (March-June, `precip_spring`): dry springs may mean less green in summer.
- **Permits:** the sum of the 12 months before the end of June, i.e. July of the year before until June (`permit_12m`). One month alone is too spiky. The first summer (2013) has no full 12 months, so it stays empty.
- **Population:** share of people aged 65+ (`share65`), the logarithm of the population (`logpop`: the size of the district, on a scale where a district twice as big is one step more) and the yearly growth (`pop_growth`).

In [ ]:
summer_weather = (climate[climate["month"].isin([7, 8])].groupby(["district_code", "year"])
                  .agg(temp_mean_ja=("temp_mean", "mean"), hot_days_ja=("hot_days", "sum")).reset_index())
spring_rain = (climate[climate["month"].between(3, 6)].groupby(["district_code", "year"])
               .agg(precip_spring=("precip_sum", "sum")).reset_index())

permits = permits.sort_values(["district_code", "year", "month"])
permits["permit_12m"] = permits.groupby("district_code")["permit_area_total_m2"].transform(lambda s: s.rolling(12).sum())
permit_june = permits.loc[permits["month"] == 6, ["district_code", "year", "permit_12m"]]  # the 12 months up to June

# The age groups from 65 on: total_65_69, total_70_74, ..., total_90plus
old_columns = [c for c in population.columns if c.startswith("total_") and c.split("_")[1][:2] in ("65", "70", "75", "80", "85", "90")]
population = population.sort_values(["district_code", "year"])
population["share65"] = population[old_columns].sum(axis=1) / population["total_pop"]
population["logpop"] = np.log(population["total_pop"])
population["pop_growth"] = population.groupby("district_code")["total_pop"].pct_change()

print("age groups counted as 65+:", [c.replace("total_", "") for c in old_columns])


## Step 3: Join everything on the district code
The greenness table is the base (one row per district and year). The static table and SEGE have one row per district, so they are joined with `many_to_one`.
The last lines add the *reliable* flag and the groupings of Step 1.

In [ ]:
df = (green
      .merge(summer_weather, on=["district_code", "year"], how="left", validate="one_to_one")
      .merge(spring_rain, on=["district_code", "year"], how="left", validate="one_to_one")
      .merge(permit_june, on=["district_code", "year"], how="left", validate="one_to_one")
      .merge(population[["district_code", "year", "total_pop", "share65", "logpop", "pop_growth"]], on=["district_code", "year"], how="left", validate="one_to_one")
      .merge(static.drop(columns=["city_code", "district_name", "city_name"]), on="district_code", how="left", validate="many_to_one")
      .merge(sege[["district_code", "sege_score", "sege_tier"]], on="district_code", how="left", validate="many_to_one")
      .merge(districts[["district_code", "center_lat", "center_lon"]], on="district_code", how="left", validate="many_to_one"))

df["permit_12m_per_built"] = df["permit_12m"] / df["built_m2_2020"]  # new floor area per m² of built-up ground (flow relative to stock)
df["metro"] = df["city_code"].isin(METRO)
df["region"] = df["city_code"].map(REGION_OF)
df["reliable"] = (df["urban_km2"] >= 0.5) & (df["landsat_cover"] >= 0.5) & df["landsat_lst_c"].notna()

assert len(df) == len(green) == 973 * 13, "the joins must not change the number of rows"
print(f"{len(df)} district-years | reliable: {df['reliable'].sum()} ({df['reliable'].mean():.0%})")
print("columns that are empty somewhere:", df.isna().sum()[lambda s: s > 0].to_dict())


## Step 4: Helpers for "within city" and the comparable rows
`minus_city_year` subtracts, for every column, the mean of the same city in the same year.
Comparing districts inside a city only makes sense if the city has at least 3 reliable districts in that year: `within` keeps only those rows (about 10,600 of the 10,657 reliable rows).

In [ ]:
def minus_city_year(table, columns):
    # value minus the mean of the same city and year: weather and regional climate drop out
    return table[columns] - table.groupby(["city_code", "year"])[columns].transform("mean")


reliable = df[df["reliable"]].copy()
cities_per_year = reliable.groupby(["city_code", "year"])["district_code"].transform("size")
within = reliable[cities_per_year >= 3].copy()
print(f"reliable rows: {len(reliable)} | rows that can be compared inside a city: {len(within)}")


# <span style="color:#F57C00">A2: Which Districts Are Left Out, and Does It Matter?</span>

The satellite value of a district is only used if its built-up area is at least 0.5 km² and at least half of its built-up pixels had a clear view. The other rows are left out of the model. The risk: if the left-out districts are special (poor, rural, in one region), the model and the planning results tell nothing about them.

## Step 1: Why are rows left out?

In [ ]:
df["status"] = np.select(
    [df["landsat_lst_c"].isna(), df["urban_km2"] < 0.5, df["landsat_cover"] < 0.5],
    ["no Landsat value", "built-up area under 0.5 km²", "under half of the pixels clear"],
    default="reliable")

counts = df["status"].value_counts()
print(counts.to_string())
counts.drop("reliable").plot.barh(figsize=(7, 2.2), color="#F57C00", title="Why district-years are left out")
plt.gca().invert_yaxis()
plt.show()


## Step 2: Who is left out? Metropolitan or not, region, development level
For every group we show the share of district-years that are left out. The SEGE tier is 1 (most developed) to 6 (least developed).

In [ ]:
df["left_out"] = ~df["reliable"]
df["metro_label"] = np.where(df["metro"], "Metropolitan", "Other provinces")

fig, axes = plt.subplots(1, 3, figsize=(16, 4), gridspec_kw={"width_ratios": [1, 2.2, 1.6]})
for ax, group, title in zip(axes, ["metro_label", "region", "sege_tier"], ["Metropolitan or not", "Region", "SEGE tier (1 = most developed)"]):
    share = df.groupby(group)["left_out"].mean().sort_values(ascending=False) if group == "region" else df.groupby(group)["left_out"].mean()
    ax.bar(share.index.astype(str), share.values, color="#F57C00")
    ax.set_title(title); ax.set_ylabel("share left out"); ax.set_ylim(0, 0.4)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## Step 3: How big are the left-out districts, and how many people live in them?
Population is the urban TÜİK population of 2025. A box plot with a logarithmic axis shows both small and big districts.

In [ ]:
latest = df[df["year"] == 2025].copy()
latest["group"] = np.where(latest["reliable"], "reliable in 2025", "left out in 2025")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=latest, x="group", y="total_pop", hue="group", palette=["#F57C00", "#7aa6c2"], ax=ax, legend=False)
ax.set_yscale("log"); ax.set_ylabel("urban population in 2025 (log scale)"); ax.set_xlabel("")
plt.show()

print("median urban population:", latest.groupby("group")["total_pop"].median().round(0).to_dict())
share_people = latest.loc[~latest["reliable"], "total_pop"].sum() / latest["total_pop"].sum()
print(f"share of the urban population that lives in left-out districts: {share_people:.1%}")

district_status = df.groupby("district_code")["reliable"].mean()
print(f"districts: never reliable {int((district_status == 0).sum())} | always reliable {int((district_status == 1).sum())} | in between {int(((district_status > 0) & (district_status < 1)).sum())}")


**What we learned (A2)**
- Most rows are left out because the **built-up area is under 0.5 km²** (1,841 of the 1,992 left-out district-years). Bad weather (clouds) hardly matters (144 rows).
- The left-out districts are **small and in the east and north**: about a third of the district-years of the Black Sea (33%) and of Eastern Anatolia (29%) are left out, against 5-8% in Marmara, the Aegean and Central Anatolia. In metropolitan provinces 12% are left out, elsewhere 20%.
- They are also **the least developed ones**: 31% (tier 5) and 30% (tier 6) of their district-years are left out, against 1-8% in tiers 1-3.
- But they hold **only 1.3% of the urban population** (median 4,851 people against 30,266). For the question "where to plant green to cool the most people" this loses little. For a statement about equity ("poor districts are hotter") it matters: a third of the poorest tiers is invisible, and the report must say so.
- 142 districts are never reliable, 742 always, 89 sometimes. The share left out is almost the same in every year (15-19%), so the rule does not bias the time series.

# <span style="color:#F57C00">B2: Greenness and Surface Temperature Over the Years</span>

From here on we only use the **reliable** rows. Three things to look at: the shape of the values, the yearly pattern (is 2025 really greener or less green?), and whether starting Landsat 9 in 2022 made a break.

## Step 1: The shape of the values

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, column, label in zip(axes, ["landsat_ndvi", "landsat_green_share", "landsat_lst_c"],
                             ["NDVI of the built-up area", "share of green pixels (NDVI ≥ 0.3)", "surface temperature (°C)"]):
    sns.histplot(reliable[column], kde=True, ax=ax, color="#7aa6c2")
    ax.set_xlabel(label)
plt.tight_layout()
plt.show()

print(reliable[["landsat_ndvi", "landsat_green_share", "landsat_lst_c"]].describe().round(2).T.to_string())


## Step 2: Yearly pattern (median and the middle half of the districts)
The line is the median of all reliable districts in a year; the band holds the middle half (25%-75%). For the green share we also show the metropolitan provinces apart from the others.

In [ ]:
def band(table, column):
    grouped = table.groupby("year")[column]
    return grouped.quantile(0.25), grouped.median(), grouped.quantile(0.75)


fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))

low, mid, high = band(reliable, "landsat_green_share")
axes[0].fill_between(mid.index, low, high, color="#7aa6c2", alpha=0.3)
axes[0].plot(mid.index, mid, color="#2c6e91", lw=2, label="all districts")
for flag, label, color in [(True, "metropolitan", "#F57C00"), (False, "other provinces", "#4a9c5d")]:
    median_by_year = reliable[reliable["metro"] == flag].groupby("year")["landsat_green_share"].median()
    axes[0].plot(median_by_year.index, median_by_year.values, color=color, lw=1.5, ls="--", label=label)
axes[0].set_title("Green share of the built-up area"); axes[0].set_ylabel("share"); axes[0].legend()

low, mid, high = band(reliable, "landsat_lst_c")
axes[1].fill_between(mid.index, low, high, color="#e8a07a", alpha=0.3)
axes[1].plot(mid.index, mid, color="#c1440e", lw=2)
axes[1].set_title("Summer surface temperature"); axes[1].set_ylabel("°C")
plt.tight_layout()
plt.show()


## Step 3: Is the yearly pattern the weather?
We take one number per year (median of the reliable districts) and put it next to the ERA5 weather. Left: green share against the rain of the spring before. Right: surface temperature against the ERA5 temperature of July + August.

In [ ]:
by_year = reliable.groupby("year").agg(green=("landsat_green_share", "median"), lst=("landsat_lst_c", "median"),
                                       spring_rain=("precip_spring", "median"), era5=("temp_mean_ja", "mean"), cover=("landsat_cover", "mean"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, x, y, xlabel, ylabel in [(axes[0], "spring_rain", "green", "rain March-June (mm, median of the districts)", "green share (median)"),
                                 (axes[1], "era5", "lst", "ERA5 temperature July-August (°C, mean)", "surface temperature (median, °C)")]:
    ax.scatter(by_year[x], by_year[y], color="#2c6e91")
    for year, row in by_year.iterrows():
        ax.annotate(str(year), (row[x], row[y]), textcoords="offset points", xytext=(4, 4), fontsize=8)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(f"correlation over the 13 years: {by_year[x].corr(by_year[y]):.2f}")
plt.tight_layout()
plt.show()
by_year.round(3)


## Step 4: Did Landsat 9 (from 2022) make a break?
For every district we take the change in surface temperature from one year to the next and show the **median change of all districts** for each pair of years. A break from the new satellite would be a big bar at 2021→2022 compared with the others; the extra clear view (the `cover` column above goes from 0.98 to 0.995) could also matter.

In [ ]:
pivot = reliable.pivot(index="district_code", columns="year", values="landsat_lst_c")
changes = pd.DataFrame({f"{year}→{year + 1}": pivot[year + 1] - pivot[year] for year in range(2013, 2025)})
median_change = changes.median()

colors = ["#c1440e" if name == "2021→2022" else "#7aa6c2" for name in median_change.index]
ax = median_change.plot.bar(figsize=(12, 3.6), color=colors)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("median change (°C)"); ax.set_title("Year-to-year change of the surface temperature, same districts")
plt.xticks(rotation=45)
plt.show()
print("2021→2022:", round(median_change["2021→2022"], 2), "| all other pairs, middle value:", round(median_change.drop("2021→2022").median(), 2), "| range of the others:", round(median_change.drop("2021→2022").min(), 2), "to", round(median_change.drop("2021→2022").max(), 2))


**What we learned (B2)**
- The values look reasonable: the median district has a green share of 0.32, an NDVI of 0.26 and a summer surface temperature of 42 °C (middle half: 40-44 °C). The green share has a long tail towards very green districts.
- **The yearly pattern is mostly the weather.** The yearly median surface temperature follows the ERA5 temperature of the same summer almost perfectly (correlation 0.93). Surface temperature also rose over the period, from about 41 °C in 2013 to 43.4 °C in 2025 (the hottest ERA5 summers are the last three).
- **The green share moves with the rain of the spring before** (correlation 0.63 over the 13 years; 0.69 without the very wet 2023). 2025 is the least green year (median 0.263) and had one of the driest springs (185 mm); 2023, the greenest of the last years, had a very wet spring (376 mm). The 2025 drop is therefore most probably dry weather, not a measuring problem; with 13 years this is a hint, not a proof.
- **No break from Landsat 9:** 2021→2022 changed by -0.6 °C, inside the range of the other years (-0.9 to +1.3 °C; the biggest change, 2022→2023, is a hot summer, not a new sensor). Metropolitan provinces are always about 0.1 less green than the others, in every year.
- For the model this means: a big part of the difference between years is **weather**, so the weather of the same summer must be a feature, and the green share of one single year is noisy (use it with care).

# <span style="color:#F57C00">C1: What Goes Together With the Surface Temperature?</span>

We look at the correlation of the surface temperature with every candidate feature, in two ways:
- **raw**: all rows together. This mixes in geography (a northern or a high district is cooler for reasons that parks cannot change);
- **within city**: after subtracting the city-year mean from the temperature and from the feature. This is the part that differs *between the districts of one city*, with the same weather.

We use the rank correlation (Spearman): it does not depend on the scale of a variable and is not distorted by extreme values.

## Step 1: Raw against within-city correlation

In [ ]:
candidates = {
    "landsat_green_share": "green share", "landsat_ndvi": "NDVI", "elevation_m": "elevation (m)", "built_share_2020": "built-up share (2020)",
    "urban_km2": "built-up area (km²)", "logpop": "size (log population)", "share65": "share aged 65+", "pop_growth": "population growth",
    "permit_12m_per_built": "permits per built-up m²", "temp_mean_ja": "ERA5 temperature Jul-Aug", "hot_days_ja": "ERA5 hot days Jul-Aug",
    "precip_spring": "rain March-June", "sege_score": "SEGE score", "center_lat": "latitude", "center_lon": "longitude",
}
columns = ["landsat_lst_c"] + list(candidates)

raw_corr = within[columns].corr(method="spearman")["landsat_lst_c"].drop("landsat_lst_c")
city_corr = minus_city_year(within, columns).corr(method="spearman")["landsat_lst_c"].drop("landsat_lst_c")
corr = pd.DataFrame({"raw": raw_corr, "within city": city_corr}).rename(index=candidates).sort_values("within city")

fig, ax = plt.subplots(figsize=(9, 6))
for position, (name, row) in enumerate(corr.iterrows()):
    ax.plot([row["raw"], row["within city"]], [position, position], color="lightgrey", zorder=1)
ax.scatter(corr["raw"], range(len(corr)), color="#7aa6c2", label="raw", zorder=2, s=50)
ax.scatter(corr["within city"], range(len(corr)), color="#F57C00", label="within city", zorder=2, s=50)
ax.axvline(0, color="black", lw=0.8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index)
ax.set_xlabel("Spearman correlation with the summer surface temperature"); ax.legend(loc="lower right")
plt.tight_layout()
plt.show()
corr.round(2)


## Step 2: The green share against the surface temperature, raw and within city
Left, all rows: every point is a district-year, and the mix of cities makes a cloud. Right, within city: now the line is clear. The slope is the change of the surface temperature for a green share that is 10 percentage points higher.

In [ ]:
within_diff = minus_city_year(within, ["landsat_lst_c", "landsat_green_share"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].hexbin(within["landsat_green_share"], within["landsat_lst_c"], gridsize=40, cmap="Blues", mincnt=1)
axes[0].set_xlabel("green share"); axes[0].set_ylabel("surface temperature (°C)"); axes[0].set_title("Raw: all rows")

axes[1].hexbin(within_diff["landsat_green_share"], within_diff["landsat_lst_c"], gridsize=40, cmap="Oranges", mincnt=1)
slope, intercept = np.polyfit(within_diff["landsat_green_share"], within_diff["landsat_lst_c"], 1)
x = np.linspace(within_diff["landsat_green_share"].min(), within_diff["landsat_green_share"].max(), 50)
axes[1].plot(x, slope * x + intercept, color="black", lw=2)
axes[1].set_xlabel("green share minus the city-year mean"); axes[1].set_ylabel("surface temperature minus the city-year mean (°C)")
axes[1].set_title(f"Within city: {slope * 0.1:.2f} °C for +10 percentage points of green")
plt.tight_layout()
plt.show()


**What we learned (C1)**
- **Green share and NDVI** are the strongest features inside a city (-0.43 and -0.42), clearly negative: greener districts are cooler. They say almost the same thing (correlation 0.97 between them), so one of them is enough in a linear model.
- **Geography fools the raw numbers.** Latitude has -0.53 raw (the north is cooler) but only -0.09 inside a city; elevation has -0.03 raw but -0.22 inside a city (the raw number hides it, probably because high places also lie in dry, hot regions). That is why the model table needs the **within-city** columns next to the raw ones.
- **The SEGE score changes its sign:** -0.10 raw, +0.08 within city. Richer districts look cooler only because they lie in cooler regions. Inside a city, the better-off districts are, if anything, a little hotter. Together with its one value per district, this is why SEGE stays **out of the model** and is only used in the risk layer.
- The share of people aged 65+ is also negative within a city (-0.21): districts with many old people are cooler. They are smaller (correlation -0.48 with size) and probably greener, not hot centers (a hint for the risk layer, not a cause).
- ERA5 weather has information *inside a city* too (hot days +0.39, mean temperature +0.36): neighbouring districts in different ERA5 cells (they differ in height) differ in weather, and the surface temperature follows.
- **Permits and population growth hardly matter for the surface temperature** (0.06 and 0.03 within city). That fits what a quick linear check showed earlier: population and permits add almost nothing to the explanation. They are kept for the risk layer, and a tree model may still find something.

# <span style="color:#F57C00">C2: Where Does the Variation Live?</span>

Every variable varies in three ways: **between cities** (climate and region), **between the districts of one city**, and **over the years** in the same district. A model can only learn how parks and buildings change the temperature from the second and third part; the first part is geography.

## Step 1: Split the variance in three parts
For a column we measure the variance of the city means (between cities), of the district means minus the city means (between districts of a city) and of what is left (over the years). The three parts add up to 100%.

In [ ]:
def variance_parts(table, column):
    value = table[column]
    city_mean = value.groupby(table["city_code"]).transform("mean")
    district_mean = value.groupby(table["district_code"]).transform("mean")
    total = value.var()
    return pd.Series({
        "between cities": city_mean.var() / total,
        "between districts of a city": (district_mean - city_mean).var() / total,
        "over the years": (value - district_mean).var() / total,
    })


parts_columns = {"landsat_lst_c": "surface temperature", "landsat_green_share": "green share", "temp_mean_ja": "ERA5 temperature Jul-Aug",
                 "elevation_m": "elevation", "built_share_2020": "built-up share", "logpop": "size (log population)", "share65": "share aged 65+",
                 "sege_score": "SEGE score", "permit_12m_per_built": "permits per built-up m²"}
parts = pd.DataFrame({label: variance_parts(reliable, column) for column, label in parts_columns.items()}).T

ax = parts.plot.barh(stacked=True, figsize=(10, 5), color=["#7aa6c2", "#F57C00", "#4a9c5d"])
ax.invert_yaxis(); ax.set_xlabel("share of the variance"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.tight_layout()
plt.show()
parts.round(2)


## Step 2: How much of the year-to-year change is shared?
The part "over the years" can be common to the whole country (a hot summer everywhere), common to one city (a hot summer there) or special to the district. We ask what share of the change of a district around its own mean is already explained by the *year*, and by the *city and year*.

In [ ]:
shared = {}
for column in ["landsat_lst_c", "landsat_green_share"]:
    around_mean = reliable[column] - reliable.groupby("district_code")[column].transform("mean")
    national = around_mean.groupby(reliable["year"]).transform("mean")
    city_year = around_mean.groupby([reliable["city_code"], reliable["year"]]).transform("mean")
    shared[column] = {"the year (all of Turkey)": national.var() / around_mean.var(), "the city and the year": city_year.var() / around_mean.var()}

pd.DataFrame(shared).rename(columns={"landsat_lst_c": "surface temperature", "landsat_green_share": "green share"}).round(2)


**What we learned (C2)**
- The surface temperature is **65% geography** (between cities), 17% between the districts of a city and 18% over the years. Green share: 59% / 34% / 7%. Elevation, the built-up share, size and the SEGE score are almost completely static: **0% over the years**.
- So the features that parks and buildings change (green share, built-up share) differ mainly **between districts**, and hardly over time. The effect of green on the temperature is therefore learned from the *comparison of districts*, not from districts that changed. **The number of really independent observations for this effect is about the number of districts (~830), not the number of rows.** More months or more rows per district (we have two summer months per year) do not change that.
- The only variable that moves a lot over time is the permits (45% over the years) and the weather. Weather does most of the work for the years: **70% of the yearly change of the surface temperature of a district is shared with the other districts of its city** (21% with all of Turkey). The model must have the weather to explain it; what is special to the district is the remaining ~30%.
- For the model: use a **cross-validation that keeps all rows of a district together** (otherwise the model recognises districts instead of learning), and give it the within-city columns.